# Automatic Differentiation: How Autograd Actually WorksWiki reference for [automatic differentiation](https://ml-viz-ruby.vercel.app/wiki/automatic-differentiation).> **Running this notebook:** in Colab, use *File → Save a copy in Drive* before editing.**The problem.** You have a scalar loss and a few million parameters, and you need$\partial L/\partial \theta_i$ for every one of them. Finite differences would cost two forwardpasses *per parameter*. Symbolic differentiation of a 50-layer composition produces an expressiontoo large to write down, and has nothing to say about `if` statements.**The idea.** Don't differentiate the *formula* — differentiate the *execution*. As the forwardpass runs, write down every operation along with a closure that converts an incoming gradient intogradients for its inputs. Then walk that record backwards. One forward pass plus one backward passgives you every gradient, regardless of how many parameters there are.**Where it shows up.** Every framework you have ever used. `loss.backward()`, `jax.grad`,`tf.GradientTape` are all this algorithm.By the end of this notebook you will have written a working autodiff engine, used it to train aneural network, and checked every gradient it produces against PyTorch.

## 0 — Setup

In [ ]:
import mathimport numpy as npimport matplotlib.pyplot as pltplt.style.use("dark_background")np.random.seed(0)print("numpy", np.__version__)

## 1 — From scratch: the tapeA framework never sees your formula. It sees a *sequence of operation calls*, and writes each onedown as it happens. That record is the **tape**.Each node needs three things:1. `v` — the forward value, because the next operation needs it.2. `src` — its parent nodes, paired with **a closure computing the local derivative**.3. `g` — a slot for the gradient, which starts at zero and gets **accumulated** into.The subtle part is point 2. The closure is created *during the forward pass* and captures theforward values it will need. `mul(a, b)` must remember `b.v` in order to report$\partial(ab)/\partial a = b$ later. **That capture is where activation memory comes from.**

In [ ]:
class Value:    "A node on the tape: a scalar, its gradient, and how to push gradient to its parents."    def __init__(self, v, src=(), op="leaf"):        self.v = float(v)        self.g = 0.0          # accumulated gradient, starts empty        self.src = src        # tuple of (parent_node, local_derivative_fn)        self.op = op          # label, for printing the tape    def __repr__(self):        return "Value(v={:.6f}, g={:.6f}, op={})".format(self.v, self.g, self.op)

### 1.1 — Operations record themselvesEvery operation does two jobs: compute the output value, and record how to send gradientbackwards. Note what each lambda closes over.- `mul` captures **both input values** (it needs the other operand for each derivative).- `tanh` captures **its own output** `t`, because $1 - t^2$ is the cheapest form of the derivative.- `add` captures **nothing** — addition just passes gradient through unchanged.Real frameworks make exactly these decisions per operation, and they are the reason differentlayers have very different activation-memory costs. ReLU stores a single sign bit per element;matmul stores both operands in full.

In [ ]:
def add(a, b):    return Value(a.v + b.v, ((a, lambda g: g), (b, lambda g: g)), "+")def mul(a, b):    # the local derivatives capture a.v and b.v RIGHT NOW, during the forward pass    return Value(a.v * b.v, ((a, lambda g: g * b.v), (b, lambda g: g * a.v)), "*")def sub(a, b):    return Value(a.v - b.v, ((a, lambda g: g), (b, lambda g: -g)), "-")def tanh(a):    t = math.tanh(a.v)    return Value(t, ((a, lambda g: g * (1.0 - t * t)),), "tanh")def relu(a):    return Value(max(0.0, a.v), ((a, lambda g: g * (1.0 if a.v > 0 else 0.0)),), "relu")

### 1.2 — Topological orderThe backward pass cannot run in an arbitrary order. If a node's output is consumed by twodifferent operations, its gradient is not final until **both** of them have contributed. Reading itearly gives a number that is well-formed and wrong.`toposort` appends each node only after all of its parents, so reversing the result gives an orderin which every consumer is processed before the node it consumed.(PyTorch uses a dependency counter instead of a depth-first sort — each node knows how manyconsumers still owe it a contribution and becomes ready when that count hits zero — but theinvariant it maintains is identical.)

In [ ]:
def toposort(out):    "Parents before children; reverse it to get a safe backward order."    order, seen = [], set()    def visit(n):        if id(n) in seen:            return        seen.add(id(n))        for parent, _ in n.src:            visit(parent)        order.append(n)      # appended only after every parent    visit(out)    return order

### 1.3 — The backward passSeed the output with $\partial L/\partial L = 1$, then walk the tape in reverse, having each nodepush gradient to its parents.The `+=` on the last line is the entire algorithm. It implements the multivariable chain rule'ssum over paths:$$\frac{\partial L}{\partial w} = \sum_i \frac{\partial L}{\partial u_i}\frac{\partial u_i}{\partial w}$$Change it to `=` and every fan-out in your graph silently returns the wrong gradient.

In [ ]:
def backward(out, accumulate=True):    "Reverse-mode AD. Set accumulate=False to see what the += is protecting you from."    order = toposort(out)    for n in order:        n.g = 0.0    out.g = 1.0    for n in reversed(order):        for parent, local in n.src:            if accumulate:                parent.g += local(n.g)     # correct            else:                parent.g = local(n.g)      # the bug    return order

### 1.4 — The case that separates autodiff from "the chain rule"Here is an expression where `w` is used **twice**:$$a = w \cdot x \qquad h = \tanh(a) \qquad o = w \cdot h \qquad d = o - t \qquad L = d^2$$One parameter, two paths to the loss. This is not a contrived example — it is exactly thestructure of **weight sharing**: an RNN reusing $W_h$ at every timestep, or a convolution reusingone kernel at every spatial position. `w`'s gradient is a *sum over paths*, and the two pathsreach it at different depths.Notice also that `L = d * d` passes the same node as both arguments, so `d` is a second, shallowerfan-out. Watch the accumulation produce the familiar factor of $2d$ without anyone writing a rulefor squaring.

In [ ]:
def build_graph(w_val, x_val=1.5, t_val=1.0):    w = Value(w_val)    x = Value(x_val)    t = Value(t_val)    a = mul(w, x)    h = tanh(a)    o = mul(w, h)          # <-- w's SECOND use: the graph is now a DAG, not a chain    d = sub(o, t)    L = mul(d, d)          # <-- d used twice as well    return w, x, t, a, h, o, d, Lw, x, t, a, h, o, d, L = build_graph(0.7)order = backward(L)print("forward pass")for name, node in zip(["w", "x", "t", "a", "h", "o", "d", "L"], [w, x, t, a, h, o, d, L]):    print("  {:>2} = {:>12.8f}   ({})".format(name, node.v, node.op))names = {id(nd): nm for nm, nd in         zip(["w", "x", "t", "a", "h", "o", "d", "L"], [w, x, t, a, h, o, d, L])}print("\ntape length:", len(order), "nodes")print("topological order :", " -> ".join(names[id(n)] for n in order))print("backward walks it :", " -> ".join(names[id(n)] for n in reversed(order)))

### 1.5 — Check it against finite differencesThe gradient is only trustworthy if something independent agrees with it. A central difference,$$\frac{\partial L}{\partial w} \approx \frac{L(w+\eta) - L(w-\eta)}{2\eta}$$is slow and imprecise, but it knows nothing about our implementation — which makes it exactly theright referee.

In [ ]:
def loss_of_w(w_val):    return build_graph(w_val)[-1].veta = 1e-6fd = (loss_of_w(0.7 + eta) - loss_of_w(0.7 - eta)) / (2 * eta)print("dL/dw  autodiff          = {:.10f}".format(w.g))print("dL/dw  finite difference = {:.10f}".format(fd))print("relative error           = {:.2e}".format(abs(w.g - fd) / abs(fd)))assert abs(w.g - fd) / abs(fd) < 1e-7print("\nthe two paths that make up w's gradient:")print("  via o (short path) = {:.10f}".format(d.g * 1.0 * h.v))print("  via a (long path)  = {:.10f}".format(a.g * x.v))print("  sum                = {:.10f}".format(d.g * h.v + a.g * x.v))

### 1.6 — What the `+=` is worthNow run the *same* code with `accumulate=False`, so the last write wins instead of thecontributions summing. This is a one-character bug.

In [ ]:
w2, *_, L2 = build_graph(0.7)backward(L2, accumulate=False)print("correct  (+=) : {:.10f}".format(w.g))print("buggy    (=)  : {:.10f}".format(w2.g))print("that is {:.1f}% of the true gradient".format(100 * w2.g / w.g))print("\nno exception. no NaN. no shape error. it just trains worse, forever.")

## 2 — The library waySame expression, PyTorch. If our engine is correct, the numbers must match to floating-pointprecision.

In [ ]:
import torchwt = torch.tensor(0.7, dtype=torch.float64, requires_grad=True)xt = torch.tensor(1.5, dtype=torch.float64)tt = torch.tensor(1.0, dtype=torch.float64)Lt = (wt * torch.tanh(wt * xt) - tt) ** 2Lt.backward()print("loss   ours = {:.12f}   torch = {:.12f}".format(L.v, Lt.item()))print("dL/dw  ours = {:.12f}   torch = {:.12f}".format(w.g, wt.grad.item()))assert np.allclose(L.v, Lt.item())assert np.allclose(w.g, wt.grad.item())print("\nmatch confirmed")

### 2.1 — Why finite differences are the fallback and not the toolAutodiff is **exact** (accurate to floating-point round-off), because it applies real derivativerules rather than approximating a limit. Finite differences are caught between two errors thatmove in opposite directions as you shrink the step:- **truncation error** $O(\eta^2)$ — shrinks as $\eta$ shrinks- **round-off error** $O(\epsilon/\eta)$ — *grows* as $\eta$ shrinks, because you are subtracting  two nearly-equal floatsThe best you can do sits at the crossover, around $\eta \approx \epsilon^{1/3}$. Autodiff has nosuch trade-off, which the plot below makes obvious.

In [ ]:
etas = np.logspace(-14, -1, 60)errs = [abs((loss_of_w(0.7 + e) - loss_of_w(0.7 - e)) / (2 * e) - w.g) / abs(w.g) for e in etas]ad_err = abs(w.g - wt.grad.item()) / abs(w.g)fig, ax = plt.subplots(figsize=(8, 4.5))ax.loglog(etas, errs, color="#f43f5e", lw=2, label="central finite difference")ax.axhline(max(ad_err, 1e-17), color="#14b8a6", lw=2, ls="--",           label="reverse-mode AD (vs torch)")best = etas[int(np.argmin(errs))]ax.axvline(best, color="#eab308", lw=1, ls=":")ax.annotate("best step $\\eta \\approx$ {:.0e}".format(best), xy=(best, min(errs)),            xytext=(best * 3, min(errs) * 300), color="#eab308", fontsize=9,            arrowprops=dict(color="#eab308", arrowstyle="->"))ax.set_xlabel("step size $\\eta$")ax.set_ylabel("relative error in $\\partial L/\\partial w$")ax.set_title("Finite differences trade truncation against round-off. Autodiff does not.")ax.legend()ax.grid(alpha=0.2)plt.tight_layout()plt.show()print("best finite-difference error: {:.2e} at eta = {:.1e}".format(min(errs), best))

**What to notice.** The V shape is the whole story. Going left from the minimum, catastrophiccancellation takes over and the estimate degrades; going right, the quadratic truncation termdominates. Even at the sweet spot you get ~8 correct digits, and you paid two forward passes for*one* parameter's derivative. Autodiff gives you ~16 digits for *all* of them in one backward pass.

## 3 — Scaling up: an array-valued engineScalar nodes make the mechanism obvious but are far too slow for real work. Real frameworks put**arrays** on the tape, so one node holds an entire layer's activations and one local derivative isa matrix operation.Only one new complication appears: **broadcasting**. If a bias of shape `(h,)` is added to a batchof shape `(n, h)`, the gradient arriving has shape `(n, h)` and must be summed back down to `(h,)`before it lands on the bias. Getting this wrong is the single most common bug when people writetheir own layers.

In [ ]:
def unbroadcast(g, shape):    "Sum a gradient back down to `shape`, undoing whatever numpy broadcast on the way forward."    while g.ndim > len(shape):        g = g.sum(axis=0)    for i, dim in enumerate(shape):        if dim == 1 and g.shape[i] != 1:            g = g.sum(axis=i, keepdims=True)    return gclass Tensor:    def __init__(self, v, src=(), op="leaf"):        self.v = np.asarray(v, dtype=np.float64)        self.g = np.zeros_like(self.v)        self.src = src        self.op = op    def __repr__(self):        return "Tensor(shape={}, op={})".format(self.v.shape, self.op)def t_add(A, B):    return Tensor(A.v + B.v,                  ((A, lambda g: unbroadcast(g, A.v.shape)),                   (B, lambda g: unbroadcast(g, B.v.shape))), "+")def t_matmul(A, B):    # the two VJPs of a matmul -- note both operands are captured    return Tensor(A.v @ B.v,                  ((A, lambda g: g @ B.v.T),                   (B, lambda g: A.v.T @ g)), "@")def t_relu(A):    mask = (A.v > 0)                      # a framework stores exactly this: one bit per element    return Tensor(A.v * mask, ((A, lambda g: g * mask),), "relu")def t_mse(pred, target):    n = pred.v.size    diff = pred.v - target    return Tensor(np.mean(diff ** 2), ((pred, lambda g: g * 2.0 * diff / n),), "mse")def t_backward(out):    order, seen = [], set()    def visit(n):        if id(n) in seen:            return        seen.add(id(n))        for p, _ in n.src:            visit(p)        order.append(n)    visit(out)    for n in order:        n.g = np.zeros_like(n.v)    out.g = np.ones_like(out.v)    for n in reversed(order):        for parent, local in n.src:            parent.g += local(n.g)        # same one line, now with arrays    return order

### 3.1 — Train a network with it, then check every gradient against PyTorchA two-layer MLP on a toy regression problem. We compute gradients with our own engine and with`torch.autograd` on identical weights, and compare **every parameter tensor**.

In [ ]:
rng = np.random.default_rng(0)n, d_in, d_hid = 64, 3, 32X = rng.normal(size=(n, d_in))y_true = np.tanh(X @ np.array([1.0, -2.0, 0.5]))[:, None] + 0.1 * rng.normal(size=(n, 1))W1 = rng.normal(scale=0.5, size=(d_in, d_hid))b1 = np.zeros(d_hid)W2 = rng.normal(scale=0.5, size=(d_hid, 1))b2 = np.zeros(1)def our_grads(W1, b1, W2, b2):    tW1, tb1, tW2, tb2 = Tensor(W1), Tensor(b1), Tensor(W2), Tensor(b2)    hid = t_relu(t_add(t_matmul(Tensor(X), tW1), tb1))    out = t_add(t_matmul(hid, tW2), tb2)    loss = t_mse(out, y_true)    t_backward(loss)    return loss.v, (tW1.g, tb1.g, tW2.g, tb2.g)def torch_grads(W1, b1, W2, b2):    ps = [torch.tensor(p, dtype=torch.float64, requires_grad=True) for p in (W1, b1, W2, b2)]    Xt = torch.tensor(X, dtype=torch.float64)    yt = torch.tensor(y_true, dtype=torch.float64)    hid = torch.relu(Xt @ ps[0] + ps[1])    out = hid @ ps[2] + ps[3]    loss = ((out - yt) ** 2).mean()    loss.backward()    return loss.item(), [p.grad.numpy() for p in ps]ours_loss, ours = our_grads(W1, b1, W2, b2)th_loss, th = torch_grads(W1, b1, W2, b2)print("loss   ours = {:.12f}   torch = {:.12f}\n".format(ours_loss, th_loss))for name, g_ours, g_th in zip(["W1", "b1", "W2", "b2"], ours, th):    err = np.abs(g_ours - g_th).max()    print("  {:>2}  shape {:<8}  max abs diff = {:.3e}".format(name, str(g_ours.shape), err))    assert np.allclose(g_ours, g_th), nameprint("\nevery gradient matches PyTorch")

### 3.2 — Does it actually learn?Gradients that match are necessary but not sufficient. Run plain gradient descent with them andwatch the loss fall.

In [ ]:
P = [W1.copy(), b1.copy(), W2.copy(), b2.copy()]lr, history = 0.05, []for step in range(2000):    loss, grads = our_grads(*P)    history.append(loss)    for i in range(4):        P[i] = P[i] - lr * grads[i]      # zero_grad is implicit: we rebuild the tape each stepfig, ax = plt.subplots(figsize=(8, 4))ax.plot(history, color="#6366f1", lw=2)ax.set_yscale("log")ax.set_xlabel("gradient descent step")ax.set_ylabel("MSE (log scale)")ax.set_title("Trained entirely with the ~70-line engine above")ax.grid(alpha=0.2)plt.tight_layout()plt.show()assert np.isfinite(history[-1]) and history[-1] < history[0] / 100, "training diverged"print("loss {:.5f} -> {:.5f}   ({:.0f}x reduction)".format(    history[0], history[-1], history[0] / history[-1]))print("label noise variance is {:.4f} -- the TRAINING loss goes below it".format(0.1 ** 2))print("because a 32-unit hidden layer has enough capacity to fit the noise too.")

**What to notice.** The training loss drops below the label-noise variance of 0.01, which is not abug — with 32 hidden units the network has enough capacity to memorize the noise as well as thesignal, which is ordinary overfitting seen from the inside.More to the point: nothing about the engine knew it was training a neural network. It recordedwhatever operations were called and replayed them. That generality is the point: the same 70 linesdifferentiate an RNN, a physics simulation, or a rendering pipeline, provided the operationsinvolved know their own local derivatives.

## 4 — Tradeoffs, cost, and when to reach for what### Forward mode vs reverse modeFor $f : \mathbb{R}^n \to \mathbb{R}^m$:| | Forward mode | Reverse mode ||---|---|---|| Computes | one column of the Jacobian per pass (JVP) | one **row** per pass (VJP) || Passes needed for the full Jacobian | $n$ | $m$ || Memory | $O(1)$ — no tape needed | $O(\text{ops})$ — must store the tape || Best when | few inputs, many outputs | **many inputs, one output** || Training a network | $10^9$ passes | **1 pass** |Training is $n = 10^9$, $m = 1$, which is the most lopsided case possible in reverse mode's favour.Reverse mode's price is the tape — and that price is real.### Where the memory goes| Cost | Scales with | Notes ||---|---|---|| Parameters | model size | what you think of as "the model" || Gradients | model size | one float per parameter || Optimizer state | 2× model size for Adam | momentum + second moment || **Activations** | **depth × batch × width** | **usually the largest of the four during training** |Activations are the tape. This is why batch size is the first thing you cut when you hit an OOM,and why [gradient checkpointing](https://ml-viz-ruby.vercel.app/wiki/gradient-checkpointing)(store $\sqrt{L}$ checkpoints, recompute the rest) buys so much memory for ~30% extra compute.### Failure modes to recognize| Symptom | Cause ||---|---|| Gradients keep growing across steps | forgot `optimizer.zero_grad()` — `+=` never resets itself || "Trying to backward through the graph a second time" | buffers are freed after `backward()`; you need `retain_graph=True` || Gradient is `None` | the tensor is not a leaf, or `requires_grad=False`, or you are inside `no_grad()` || "a variable needed for gradient computation has been modified" | an in-place op overwrote a value some node captured for its backward || Gradient is exactly 0 and shouldn't be | ReLU/`abs` at a kink, or a `round`/`argmax` in the path || Silently wrong gradients on a shared weight | a hand-written backward that assigns instead of accumulating |<br>**Autodiff is exact, but only of the code you ran.** It differentiates the program, not yourintent. Where the program is not differentiable it still returns a number: `relu` at exactly 0 andevery `floor`/`round`/`argmax` hands back a convenient subgradient with no warning. Straight-throughestimators exist to put a *useful* fiction there instead of the default one.

## 5 — Your turnThree exercises. Each has a `# TODO(you)` outline, an assertion that passes silently when you getit right, and a collapsed solution.

### Exercise 1 — Add `exp` and `div` to the scalar engineImplement two more operations. Remember that each must return a `Value` whose `src` pairs everyparent with its local derivative, and that the derivative closures capture forward values.- $\frac{d}{da}e^a = e^a$ (capture the output, like `tanh` does)- $\frac{\partial}{\partial a}\frac{a}{b} = \frac{1}{b}$ and $\frac{\partial}{\partial b}\frac{a}{b} = -\frac{a}{b^2}$

In [ ]:
def exp(a):    # TODO(you): compute e = math.exp(a.v), then return a Value whose local    # derivative multiplies the incoming gradient by e    raise NotImplementedErrordef div(a, b):    # TODO(you): two parents, two different local derivatives    raise NotImplementedError# --- check (should print nothing but "passed") ---try:    p = Value(0.4)    q = Value(1.7)    out = div(exp(p), q)    backward(out)    fp = lambda pv: math.exp(pv) / 1.7    fq = lambda qv: math.exp(0.4) / qv    assert abs(p.g - (fp(0.4 + 1e-6) - fp(0.4 - 1e-6)) / 2e-6) < 1e-6    assert abs(q.g - (fq(1.7 + 1e-6) - fq(1.7 - 1e-6)) / 2e-6) < 1e-6    print("passed")except NotImplementedError:    print("not implemented yet")

<details><summary>Solution</summary>```pythondef exp(a):    e = math.exp(a.v)    return Value(e, ((a, lambda g: g * e),), "exp")def div(a, b):    return Value(a.v / b.v,                 ((a, lambda g: g / b.v),                  (b, lambda g: -g * a.v / (b.v ** 2))), "/")````exp` captures its own output, exactly as `tanh` captures `t` — the derivative of the exponentialis the exponential, so there is nothing cheaper to store.</details>

### Exercise 2 — Count how many times each node is usedWrite a function that returns, for every node in a graph, how many other nodes consume its output.This is the *out-degree*, and it is precisely the count PyTorch tracks to decide when a node'sgradient is final.For the graph in section 1.4, `w` should have out-degree 2 and `d` should have out-degree 2.

In [ ]:
def out_degrees(out):    # TODO(you): walk the tape (toposort helps) and count, for each node,    # how many times it appears as somebody's parent.    # Return a dict mapping id(node) -> count.    raise NotImplementedErrortry:    w3, x3, t3, a3, h3, o3, d3, L3 = build_graph(0.7)    deg = out_degrees(L3)    assert deg[id(w3)] == 2, "w feeds both a and o"    assert deg[id(d3)] == 2, "d is used twice by L = d*d"    assert deg[id(h3)] == 1    assert deg[id(L3)] == 0, "the output is consumed by nobody"    print("passed -- fan-out nodes:",          [nm for nm, nd in zip("w x t a h o d L".split(),                                [w3, x3, t3, a3, h3, o3, d3, L3]) if deg[id(nd)] > 1])except NotImplementedError:    print("not implemented yet")

<details><summary>Solution</summary>```pythondef out_degrees(out):    deg = {}    for n in toposort(out):        deg.setdefault(id(n), 0)        for parent, _ in n.src:            deg[id(parent)] = deg.get(id(parent), 0) + 1    return deg```Any node with a count above 1 is a fan-out, and is a node whose gradient would be wrong if thebackward pass overwrote instead of accumulating.</details>

### Exercise 3 — Measure the tapeWrite a function that runs a forward pass through an $L$-layer MLP and reports how many arrayelements the tape is holding alive. Then confirm it grows **linearly in depth** — the fact thatmotivates gradient checkpointing.

In [ ]:
def tape_elements(depth, width=64, batch=32, seed=0):    # TODO(you):    #   1. build `depth` layers of (width x width) weights    #   2. run x -> relu(x @ W) repeatedly, keeping the Tensors    #   3. t_backward on a t_mse against zeros    #   4. return the total number of elements across every node's .v on the tape    raise NotImplementedErrortry:    sizes = {L_: tape_elements(L_) for L_ in (2, 4, 8, 16)}    for L_, s in sizes.items():        print("depth {:>2}: {:>8,} elements on the tape".format(L_, s))    growth = (sizes[16] - sizes[8]) / (sizes[8] - sizes[4])    assert 1.8 < growth < 2.2, "doubling the depth should roughly double the increment"    print("\nlinear in depth, as expected -- this is what checkpointing attacks")except NotImplementedError:    print("not implemented yet")

<details><summary>Solution</summary>```pythondef tape_elements(depth, width=64, batch=32, seed=0):    r = np.random.default_rng(seed)    xs = Tensor(r.normal(size=(batch, width)))    node = xs    for _ in range(depth):        W = Tensor(r.normal(scale=1.0 / np.sqrt(width), size=(width, width)))        node = t_relu(t_matmul(node, W))    loss = t_mse(node, np.zeros((batch, width)))    order = t_backward(loss)    return sum(int(n.v.size) for n in order)```Each layer contributes a fixed number of elements (its weights, its matmul output and its ReLUoutput), so the total is affine in depth. Batch size multiplies the activation part but not theweight part, which is why cutting the batch is the quickest way out of an OOM.</details>

## 6 — Key takeaways- **Autodiff differentiates the execution, not the formula.** Operations record themselves onto a  tape as they run, each storing a closure that converts an incoming gradient into gradients for  its inputs.- **Those closures capture forward values, and that capture is your activation memory.** It scales  with depth × batch × width, and during training it is usually larger than the parameters,  gradients and optimizer state.- **Reverse topological order is what makes fan-out safe.** A node's gradient must not be read  until every consumer has contributed.- **`+=` is the multivariable chain rule.** It is why weight sharing needs no special support, why  `L = d*d` yields $2d$ for free, and why `zero_grad()` is your responsibility.- **Reverse mode is $O(1)$ passes in the number of parameters and $O(m)$ in the number of outputs.**  Training is $m = 1$, which is the best case; that asymmetry is what makes deep learning affordable.- **It is exact but literal.** It differentiates the program that ran, kinks and rounding  operations included.**Where to go next:**- [Chain Rule and Backpropagation](https://ml-viz-ruby.vercel.app/courses/calculus-for-ml/02-chain-rule-and-backpropagation) — the mathematics this implements- [Jacobians & Vector Calculus](https://ml-viz-ruby.vercel.app/courses/calculus-for-ml/04-jacobians) — VJPs, which are what each backward closure computes- [Gradient Checkpointing](https://ml-viz-ruby.vercel.app/wiki/gradient-checkpointing) — trading compute for the tape's memory- [BPTT](https://ml-viz-ruby.vercel.app/wiki/bptt-algorithm) — fan-out at scale, one edge per timestep